# Component 3 - Phase 4: scale calibration (um per pixel)

The final StarDist model + Phase 3 extraction on the **69 calibration images** (reserved in Phase 0,
used for nothing else) -> um/px = median of sqrt(Konan mean area / our mean area).
Then a check on the 197 validation images (ECD vs Konan). Rules fixed in `config.yaml` (`calibration:`).

**Runtime > Change runtime type > L4 GPU** (cell 3, ~2 min); the rest also runs on CPU.

In [ ]:
# 1. Setup (mount Drive, pull latest code, copy images, set paths)
from google.colab import drive; drive.mount("/content/drive")
REPO_DIR = "/content/drive/MyDrive/Colab Notebooks/Research_Project/xai-cornea-grading"
!git -C "{REPO_DIR}" pull --ff-only
%run "{REPO_DIR}/ml/cellular/colab_setup.py"

In [ ]:
# 2. Packages (Colab only) + unit tests
!pip -q install tensorflow tf-keras stardist
!nvidia-smi -L
!python -m pytest tests/test_calibration.py tests/test_extraction.py tests/test_normalise.py -q

In [ ]:
# 3. Segment the 69 calibration images with the final StarDist model (lit-tissue normalisation)
!python -m segmentation.infer_full --run stardist_finetune_flat --split calibration 2>&1 | grep -E "images|done|Error|error"

In [ ]:
# 4. Phase 3 extraction on the calibration images (CPU work)
!python -m extraction.run_extract --split calibration

In [ ]:
# 5. Step 4.1 - the scale: median of per-image um/px, 95% bootstrap CI, sensitivity rows
!python -m calibration.scale

In [ ]:
# 6. Step 4.2 - check on the 197 validation images: ECD vs Konan (bias, MAPE, Bland-Altman)
!python -m calibration.scale --check

In [ ]:
# 7. Figure: per-image scale (calibration) + Bland-Altman (validation)
from IPython.display import Image, display
!python -m calibration.scale --figure
display(Image(f"{OUTPUT_ROOT}/calibration/stardist_finetune_flat/figures/calibration.png", width=1100))